# 🚀 Apigee X Provisioning - Global Control Plane (PSC or VPC Peering)

<a href="https://colab.research.google.com/github/luiscuellarh/apigee-unified-ai-gateway/blob/main/01_apigee_provisioning_global.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook automates end-to-end provisioning of an **Apigee X** organization on the **Global Control Plane**, supporting both **Private Service Connect (PSC)** (`"disableVpcPeering": "true"`) and **VPC Network Peering** (`"disableVpcPeering": "false"`).

---

### 🌟 Why this Notebook instead of the Apigee Provisioning Wizard?
1. **Explicit Networking Mode Control (`PSC` vs `VPC_PEERING`)**: Choose between modern **Private Service Connect (PSC)** with zero `/22` IP CIDR reservations, or traditional **Service Networking VPC Peering** with automated pre-flight RFC1918 & CIDR overlap checks.
2. **No Fragile Envoy VM Bridges in PSC Mode**: Native Google Cloud PSC Load Balancing via PSC Service Attachments and PSC Network Endpoint Groups (NEGs).
3. **Automated SSL & Wildcard Hostname**: Automatically configures Google-managed SSL certificates and free `nip.io` wildcard DNS so your HTTPS endpoint is ready to test immediately.

---

### ⚠️ Critical Decisions Before You Begin

> [!WARNING]
> **Billing Type is Permanent & Cannot Be Changed Later!**
> - **`EVALUATION` (Trial)**: 60-day temporary trial org. **Cannot be upgraded or converted to a paid org later!** Not covered by Google Cloud SLAs and not supported for production.
> - **`PAYG` (Pay-As-You-Go)**: Standard production-ready pay-as-you-go billing based on API calls and analytics.
> - **`SUBSCRIPTION`**: For enterprise annual/multi-year subscription commitments.
> 
> *If you are planning for eventual production use, select `PAYG` or `SUBSCRIPTION`.*

---

### 🏗️ Architecture Overview
```
Client (Internet)
       │
       ▼
[Global External HTTPS Load Balancer] (Port 443 + Google-managed SSL)
       │
       ├──► (PSC Mode) ─────────► [PSC NEG] ──► [Apigee Service Attachment]
       │                                                 │
       └──► (VPC Peering Mode) ─► [Service Networking] ──┼──► [Apigee Runtime Instance (Tenant Project)]
```


## Step 0: Colab Authentication & Configuration

Run the cell below to authenticate your Google Cloud account and configure your deployment parameters.


In [ ]:
#@title ⚙️ Configuration Form [Run this cell]
#@markdown Fill in your configuration parameters below:

import os
import sys
import json
import time
import ipaddress
import subprocess
import requests

# Authenticate in Google Colab
try:
    from google.colab import auth
    auth.authenticate_user()
    print("✅ Authenticated successfully with Google Cloud in Colab.")
except ImportError:
    print("ℹ️ Running in local/standard Jupyter environment.")

#@markdown #### 1. Google Cloud Project & Region
PROJECT_ID = "your-project-id" #@param {type:"string"}
REGION = "europe-west1" #@param ["europe-west1", "europe-west3", "europe-west4", "us-central1", "us-east1", "us-east4", "us-west1", "asia-east1", "asia-northeast1", "asia-southeast1", "australia-southeast1"]
ANALYTICS_REGION = "europe-west1" #@param ["europe-west1", "europe-west3", "europe-west4", "us-central1", "us-east1", "us-east4", "us-west1", "asia-east1", "asia-northeast1", "asia-southeast1", "australia-southeast1"]

#@markdown #### 2. Billing & Runtime Configuration
#@markdown *Important: EVALUATION cannot be upgraded to paid/production later.*
BILLING_TYPE = "EVALUATION" #@param ["EVALUATION", "PAYG", "SUBSCRIPTION"]
RUNTIME_TYPE = "CLOUD" #@param ["CLOUD"]

#@markdown #### 3. Networking Mode & VPC Configuration
#@markdown *Important: The networking mode (`PSC` vs `VPC_PEERING`) is fixed at Apigee organization creation and cannot be changed afterwards.*
#@markdown *If you change any of the network, subnet, or range names below, the values must match whatever was used in the networking notebook.*
NETWORKING_MODE = "PSC" #@param ["PSC", "VPC_PEERING"]
#@markdown **Shared VPC & Regional Subnet (Required in BOTH modes — `PSC` consumes `NETWORK_NAME` for the PSC NEG; `VPC_PEERING` uses it as the organization's `authorizedNetwork`):**
NETWORK_NAME = "apigee-network" #@param {type:"string"}
SUBNET_NAME = "apigee-subnet" #@param {type:"string"}
SUBNET_CIDR = "10.10.0.0/24" #@param {type:"string"}
#@markdown ---
#@markdown **VPC Peering Reserved Ranges & Allocation Parameters (Used ONLY when `NETWORKING_MODE == "VPC_PEERING"` — note that the support range is always `/28` regardless of `PEERING_CIDR_SIZE`):**
PEERING_RANGE_NAME = "apigee-peering-range" #@param {type:"string"}
PEERING_CIDR_SIZE = "/22" #@param ["/22", "/28"]
PEERING_RANGE_BASE_IP = "10.20.0.0" #@param {type:"string"}
SUPPORT_RANGE_NAME = "apigee-support-range" #@param {type:"string"}
SUPPORT_RANGE_BASE_IP = "10.30.0.0" #@param {type:"string"}
AUTO_ALLOCATE = False #@param {type:"boolean"}

SUPPORT_PREFIX_LEN = 28
PEERING_PREFIX_LEN = int(PEERING_CIDR_SIZE.lstrip("/"))

#@markdown #### 4. Apigee Environment & Instance Configuration
#@markdown *Leave INSTANCE_NAME empty or 'auto' to default to apigee-{REGION}*
INSTANCE_NAME = "auto" #@param {type:"string"}
ENV_NAME = "dev" #@param {type:"string"}
ENVGROUP_NAME = "dev-group" #@param {type:"string"}

# Compute default instance name if auto/empty
if not INSTANCE_NAME.strip() or INSTANCE_NAME.lower() == "auto":
    INSTANCE_NAME = f"apigee-{REGION}"

#@markdown #### 5. Optional Add-ons
ENABLE_MONETIZATION = True #@param {type:"boolean"}
ENABLE_API_SECURITY = True #@param {type:"boolean"}
ENABLE_ADVANCED_APIOPS = True #@param {type:"boolean"}

# Validate PROJECT_ID
if PROJECT_ID == "your-project-id" or not PROJECT_ID.strip():
    raise ValueError("❌ Please provide a valid Google Cloud PROJECT_ID in the form above!")

# Validate NETWORKING_MODE
if NETWORKING_MODE not in ("PSC", "VPC_PEERING"):
    raise ValueError("❌ Please select a valid NETWORKING_MODE ('PSC' or 'VPC_PEERING') in the form above!")

# Set environment variables for shell commands
os.environ["PROJECT_ID"] = PROJECT_ID
os.environ["REGION"] = REGION
os.environ["ANALYTICS_REGION"] = ANALYTICS_REGION
os.environ["BILLING_TYPE"] = BILLING_TYPE
os.environ["RUNTIME_TYPE"] = RUNTIME_TYPE
os.environ["NETWORKING_MODE"] = NETWORKING_MODE
os.environ["NETWORK_NAME"] = NETWORK_NAME
os.environ["SUBNET_NAME"] = SUBNET_NAME
os.environ["SUBNET_CIDR"] = SUBNET_CIDR
os.environ["PEERING_RANGE_NAME"] = PEERING_RANGE_NAME
os.environ["PEERING_CIDR_SIZE"] = PEERING_CIDR_SIZE
os.environ["PEERING_PREFIX_LEN"] = str(PEERING_PREFIX_LEN)
os.environ["PEERING_RANGE_BASE_IP"] = PEERING_RANGE_BASE_IP
os.environ["SUPPORT_RANGE_NAME"] = SUPPORT_RANGE_NAME
os.environ["SUPPORT_PREFIX_LEN"] = str(SUPPORT_PREFIX_LEN)
os.environ["SUPPORT_RANGE_BASE_IP"] = SUPPORT_RANGE_BASE_IP
os.environ["AUTO_ALLOCATE"] = str(AUTO_ALLOCATE)
os.environ["INSTANCE_NAME"] = INSTANCE_NAME
os.environ["ENV_NAME"] = ENV_NAME
os.environ["ENVGROUP_NAME"] = ENVGROUP_NAME

# Set gcloud default project
!gcloud config set project $PROJECT_ID --quiet

print(f"\n🎯 Configured for Project: {PROJECT_ID}")
print(f"🌍 Instance: {INSTANCE_NAME} in Region: {REGION} | Analytics: {ANALYTICS_REGION}")
print(f"💳 Billing Type: {BILLING_TYPE} | Runtime Type: {RUNTIME_TYPE}")
print(f"🔌 Networking Mode: {NETWORKING_MODE}")
print(f"🌐 VPC Network: {NETWORK_NAME} | Regional Subnet: {SUBNET_NAME} ({SUBNET_CIDR})")
print(
    f"📐 Peering Config (VPC_PEERING only): Runtime={PEERING_RANGE_NAME} ({PEERING_RANGE_BASE_IP}{PEERING_CIDR_SIZE}), "
    f"Support={SUPPORT_RANGE_NAME} ({SUPPORT_RANGE_BASE_IP}/{SUPPORT_PREFIX_LEN}), AutoAllocate={AUTO_ALLOCATE}"
)
print(f"✨ Addons: Monetization={ENABLE_MONETIZATION}, ApiSecurity={ENABLE_API_SECURITY}, AdvancedApiOps={ENABLE_ADVANCED_APIOPS}")


In [ ]:
# Helper function for Google API requests and long-running operation polling
def get_auth_token():
    token = subprocess.check_output(
        ["gcloud", "auth", "print-access-token"], text=True
    ).strip()
    return token

def apigee_headers():
    return {
        "Authorization": f"Bearer {get_auth_token()}",
        "Content-Type": "application/json; charset=utf-8"
    }

def poll_apigee_operation(op_url, description="Operation", poll_interval=15, max_timeout=3600):
    start_time = time.time()
    print(f"⏳ Waiting for {description} to complete...")
    while time.time() - start_time < max_timeout:
        resp = requests.get(op_url, headers=apigee_headers())
        if resp.status_code == 200:
            data = resp.json()
            if data.get("done", False):
                if "error" in data:
                    raise RuntimeError(f"❌ {description} failed: {json.dumps(data['error'], indent=2)}")
                elapsed = int(time.time() - start_time)
                print(f"✅ {description} completed successfully in {elapsed // 60}m {elapsed % 60}s!")
                return data.get("response", {})
        elapsed = int(time.time() - start_time)
        mins, secs = divmod(elapsed, 60)
        print(f"   ⏱️ Still in progress... ({mins}m {secs:02d}s elapsed)", end="\r")
        time.sleep(poll_interval)
    raise TimeoutError(f"❌ Timed out waiting for {description} after {max_timeout // 60} minutes.")

print("🛠️ Apigee helper utilities initialized.")


## Step 1: Enable Google Cloud APIs

We enable the required APIs:
- **Apigee API** (`apigee.googleapis.com`)
- **Compute Engine API** (`compute.googleapis.com`)
- **API Hub** (`apihub.googleapis.com`)
- **Cloud KMS** (`cloudkms.googleapis.com`)
- **Service Networking API** (`servicenetworking.googleapis.com`) — *enabled only when `NETWORKING_MODE == "VPC_PEERING"`*


In [ ]:
#@title 🔌 Enable APIs
apis_to_enable = [
    "apigee.googleapis.com",
    "compute.googleapis.com",
    "apihub.googleapis.com",
    "cloudkms.googleapis.com",
]

if NETWORKING_MODE == "VPC_PEERING":
    apis_to_enable.append("servicenetworking.googleapis.com")
    print("ℹ️ VPC_PEERING mode selected — including servicenetworking.googleapis.com.")
elif NETWORKING_MODE == "PSC":
    print("ℹ️ PSC mode selected — skipping servicenetworking.googleapis.com.")
else:
    raise ValueError(f"❌ Unrecognised NETWORKING_MODE: '{NETWORKING_MODE}'. Expected 'PSC' or 'VPC_PEERING'.")

apis_str = " ".join(apis_to_enable)
!gcloud services enable $apis_str --project $PROJECT_ID

print(f"✅ Enabled Google Cloud APIs: {', '.join(apis_to_enable)}")


## Step 2: Networking Mode — PSC or VPC Peering

Review the architectural comparison below **before** selecting your networking mode:

| Dimension | PSC (`disableVpcPeering: true`) | VPC Peering (`disableVpcPeering: false`) |
| :--- | :--- | :--- |
| **IP ranges consumed** | None (0 peering CIDRs; uses only a regional subnet for PSC NEGs/endpoints) | Requires a `/22` (prod) or `/28` (eval) runtime range **plus** a `/28` support range |
| **Resizable after org creation** | N/A — no peering ranges to resize; PSC endpoints scale elastically | **No** — reserved peering ranges cannot be resized in place once allocated |
| **Service Networking required** | **No** (`servicenetworking.googleapis.com` is not used) | **Yes** (`servicenetworking.googleapis.com` VPC peering connection required) |
| **Backend reachability model** | Point-to-point attachment via PSC Service Attachment (`serviceAttachment`) & PSC NEGs | Transitive-free L3 VPC network peering directly to the Apigee runtime IP (`host`) |
| **Typical use case** | Modern enterprise deployments, overlapping/exhausted RFC1918 space, multi-VPC hub-and-spoke | Legacy topologies requiring direct private RFC1918 routing over Service Networking |

---

> [!WARNING]
> **⚠️ WARNING: Networking Mode is Fixed at Org Creation!**
> The choice between `PSC` (`disableVpcPeering: true`) and `VPC_PEERING` (`disableVpcPeering: false`) is permanently locked when the Apigee Organization is created and **cannot be changed afterwards** without deleting and recreating the organization.

> [!WARNING]
> **⚠️ WARNING: Peering Ranges Cannot Be Resized in Place!**
> In `VPC_PEERING` mode, reserved peering ranges cannot be expanded or shrunk in place. Use **`/22`** for production-grade instances (supports multi-region expansion and high-scale pods); **`/28`** is strictly for minimal-footprint / evaluation environments.

> [!WARNING]
> **⚠️ WARNING: `/22` and `/28` Peering Ranges Are RESERVATIONS, Not Subnets!**
> The runtime (`/22` or `/28`) and support (`/28`) ranges are **global, regionless IP address reservations** (`--purpose=VPC_PEERING`) delegated to Google's Service Networking tenant project. They are **NOT subnets** in your VPC. **Never create a subnetwork inside these reserved CIDR blocks.**


In [ ]:
#@title 🌐 Networking Mode & VPC Configuration [Run this cell]
#@markdown All networking configuration parameters (`NETWORKING_MODE`, VPC/subnet names, and VPC peering range settings) are defined in **Step 0**. Run Step 0 first if you need to modify any parameter.

# Fail-fast guard: ensure all required configuration variables from Step 0 are defined
_required_step0_vars = [
    "PROJECT_ID",
    "REGION",
    "NETWORKING_MODE",
    "NETWORK_NAME",
    "SUBNET_NAME",
    "SUBNET_CIDR",
    "PEERING_RANGE_NAME",
    "PEERING_CIDR_SIZE",
    "PEERING_PREFIX_LEN",
    "PEERING_RANGE_BASE_IP",
    "SUPPORT_RANGE_NAME",
    "SUPPORT_PREFIX_LEN",
    "SUPPORT_RANGE_BASE_IP",
    "AUTO_ALLOCATE",
]
_missing_step0_vars = [v for v in _required_step0_vars if v not in globals()]
if _missing_step0_vars:
    raise RuntimeError(
        f"❌ Missing required configuration variable(s): {', '.join(_missing_step0_vars)}. "
        "Please run Step 0 (Colab Authentication & Configuration) first!"
    )

print(f"ℹ️ Using Networking Mode from Step 0: {NETWORKING_MODE}")

# ======================================================================
# SHARED (both modes): Verify/Create VPC Network (CUSTOM mode) & Subnet
# ======================================================================
net_check = subprocess.run(
    ["gcloud", "compute", "networks", "describe", NETWORK_NAME,
     f"--project={PROJECT_ID}", "--format=json"],
    capture_output=True, text=True
)

if net_check.returncode != 0:
    print(f"ℹ️ VPC network '{NETWORK_NAME}' does not exist in project '{PROJECT_ID}'. Creating in CUSTOM subnet mode...")
    subprocess.run(
        ["gcloud", "compute", "networks", "create", NETWORK_NAME,
         f"--project={PROJECT_ID}", "--subnet-mode=custom", "--quiet"],
        check=True
    )
    print(f"✅ Created VPC network '{NETWORK_NAME}' (custom subnet mode).")
else:
    print(f"✅ VPC network '{NETWORK_NAME}' exists and is ready to use.")

# Verify or create the deployed regional subnet in REGION
subnet_check = subprocess.run(
    ["gcloud", "compute", "networks", "subnets", "describe", SUBNET_NAME,
     f"--region={REGION}", f"--project={PROJECT_ID}", "--format=json"],
    capture_output=True, text=True
)

if subnet_check.returncode == 0:
    subnet_info = json.loads(subnet_check.stdout)
    DEPLOYED_SUBNET_CIDR = subnet_info.get("ipCidrRange", SUBNET_CIDR)
    print(f"✅ Regional subnet '{SUBNET_NAME}' already exists in '{REGION}' with CIDR {DEPLOYED_SUBNET_CIDR}.")
else:
    # Check if an auto-mode subnet already exists in REGION for NETWORK_NAME
    existing_region_subnets = subprocess.run(
        ["gcloud", "compute", "networks", "subnets", "list",
         f"--network={NETWORK_NAME}", f"--regions={REGION}",
         f"--project={PROJECT_ID}", "--format=json"],
        capture_output=True, text=True, check=True
    )
    region_subnet_list = json.loads(existing_region_subnets.stdout or "[]")
    if region_subnet_list:
        DEPLOYED_SUBNET_CIDR = region_subnet_list[0]["ipCidrRange"]
        print(f"✅ Using existing regional subnet '{region_subnet_list[0]['name']}' in '{REGION}' with CIDR {DEPLOYED_SUBNET_CIDR}.")
    else:
        print(f"ℹ️ Creating regional subnet '{SUBNET_NAME}' ({SUBNET_CIDR}) in region '{REGION}'...")
        subprocess.run(
            ["gcloud", "compute", "networks", "subnets", "create", SUBNET_NAME,
             f"--network={NETWORK_NAME}", f"--region={REGION}",
             f"--range={SUBNET_CIDR}", f"--project={PROJECT_ID}", "--quiet"],
            check=True
        )
        DEPLOYED_SUBNET_CIDR = SUBNET_CIDR
        print(f"✅ Created regional subnet '{SUBNET_NAME}' in '{REGION}' with CIDR {DEPLOYED_SUBNET_CIDR}.")

os.environ["DEPLOYED_SUBNET_CIDR"] = DEPLOYED_SUBNET_CIDR

# Converged variables populated by both branches
ALLOCATED_PEERING_CIDR = "N/A (PSC)"
ALLOCATED_SUPPORT_CIDR = "N/A (PSC)"
PEERING_STATE = "N/A (PSC)"

# ======================================================================
# MODE-SPECIFIC BRANCHING
# ======================================================================
if NETWORKING_MODE == "PSC":
    print("ℹ️ PSC mode — no peering ranges reserved, no Service Networking connection required.")

elif NETWORKING_MODE == "VPC_PEERING":
    # Ensure servicenetworking.googleapis.com is enabled
    print("ℹ️ Ensuring servicenetworking.googleapis.com API is enabled...")
    subprocess.run(
        ["gcloud", "services", "enable", "servicenetworking.googleapis.com",
         f"--project={PROJECT_ID}", "--quiet"],
        check=True
    )
    print("✅ servicenetworking.googleapis.com is enabled.")

    # ------------------------------------------------------------------
    # a) Pre-flight validation (RFC1918 + CIDR non-overlap checks)
    # ------------------------------------------------------------------
    print("ℹ️ Running pre-flight IP and CIDR overlap validation...")
    rfc1918_blocks = [
        ipaddress.ip_network("10.0.0.0/8"),
        ipaddress.ip_network("172.16.0.0/12"),
        ipaddress.ip_network("192.168.0.0/16"),
    ]

    def is_rfc1918(net):
        return net.version == 4 and any(net.subnet_of(block) for block in rfc1918_blocks)

    try:
        subnet_net = ipaddress.ip_network(DEPLOYED_SUBNET_CIDR, strict=False)
    except ValueError as exc:
        raise ValueError(f"❌ Invalid regional subnet CIDR '{DEPLOYED_SUBNET_CIDR}': {exc}") from exc

    # Query existing subnets in the project
    subnets_proc = subprocess.run(
        ["gcloud", "compute", "networks", "subnets", "list",
         f"--network={NETWORK_NAME}", f"--project={PROJECT_ID}", "--format=json"],
        capture_output=True, text=True, check=True
    )
    existing_subnets = json.loads(subnets_proc.stdout or "[]")

    # Query existing global reserved ranges in the project
    addrs_proc = subprocess.run(
        ["gcloud", "compute", "addresses", "list", "--global",
         f"--project={PROJECT_ID}", "--format=json"],
        capture_output=True, text=True, check=True
    )
    existing_global_addrs = json.loads(addrs_proc.stdout or "[]")

    if not AUTO_ALLOCATE:
        try:
            peering_ip = ipaddress.ip_address(PEERING_RANGE_BASE_IP.strip())
            runtime_net = ipaddress.ip_network(f"{peering_ip}/{PEERING_PREFIX_LEN}", strict=True)
        except ValueError as exc:
            raise ValueError(
                f"❌ Pre-flight validation failed for PEERING_RANGE_BASE_IP='{PEERING_RANGE_BASE_IP}' "
                f"with prefix '/{PEERING_PREFIX_LEN}': {exc}"
            ) from exc

        try:
            support_ip = ipaddress.ip_address(SUPPORT_RANGE_BASE_IP.strip())
            support_net = ipaddress.ip_network(f"{support_ip}/{SUPPORT_PREFIX_LEN}", strict=True)
        except ValueError as exc:
            raise ValueError(
                f"❌ Pre-flight validation failed for SUPPORT_RANGE_BASE_IP='{SUPPORT_RANGE_BASE_IP}' "
                f"with prefix '/{SUPPORT_PREFIX_LEN}': {exc}"
            ) from exc

        if not is_rfc1918(runtime_net):
            raise ValueError(f"❌ Runtime peering range {runtime_net} is not a valid private RFC1918 IPv4 range!")
        if not is_rfc1918(support_net):
            raise ValueError(f"❌ Support peering range {support_net} is not a valid private RFC1918 IPv4 range!")

        # Mutual overlap checks between runtime range, support range, and deployed regional subnet
        if runtime_net.overlaps(support_net):
            raise ValueError(
                f"❌ CIDR Overlap: Runtime peering range ({runtime_net}) overlaps with "
                f"Support peering range ({support_net})!"
            )
        if runtime_net.overlaps(subnet_net):
            raise ValueError(
                f"❌ CIDR Overlap: Runtime peering range ({runtime_net}) overlaps with "
                f"regional subnet '{SUBNET_NAME}' ({subnet_net})!"
            )
        if support_net.overlaps(subnet_net):
            raise ValueError(
                f"❌ CIDR Overlap: Support peering range ({support_net}) overlaps with "
                f"regional subnet '{SUBNET_NAME}' ({subnet_net})!"
            )

        # Check against all existing subnets in the VPC
        for s in existing_subnets:
            s_cidr = s.get("ipCidrRange")
            if not s_cidr:
                continue
            s_net = ipaddress.ip_network(s_cidr, strict=False)
            s_name = s.get("name", "unknown")
            if runtime_net.overlaps(s_net):
                raise ValueError(
                    f"❌ CIDR Overlap: Runtime peering range ({runtime_net}) overlaps with "
                    f"existing subnet '{s_name}' ({s_net})!"
                )
            if support_net.overlaps(s_net):
                raise ValueError(
                    f"❌ CIDR Overlap: Support peering range ({support_net}) overlaps with "
                    f"existing subnet '{s_name}' ({s_net})!"
                )

        # Check against all existing global reserved address ranges in the project
        for addr in existing_global_addrs:
            a_name = addr.get("name", "")
            if a_name in (PEERING_RANGE_NAME, SUPPORT_RANGE_NAME):
                continue
            a_ip = addr.get("address")
            a_plen = addr.get("prefixLength")
            if a_ip and a_plen is not None:
                a_net = ipaddress.ip_network(f"{a_ip}/{a_plen}", strict=False)
                if runtime_net.overlaps(a_net):
                    raise ValueError(
                        f"❌ CIDR Overlap: Runtime peering range ({runtime_net}) overlaps with "
                        f"existing global reserved range '{a_name}' ({a_net})!"
                    )
                if support_net.overlaps(a_net):
                    raise ValueError(
                        f"❌ CIDR Overlap: Support peering range ({support_net}) overlaps with "
                        f"existing global reserved range '{a_name}' ({a_net})!"
                    )
        print(f"✅ Pre-flight validation passed: {runtime_net} and {support_net} are valid RFC1918 and conflict-free.")
    else:
        print("ℹ️ AUTO_ALLOCATE=True — Google Cloud will automatically select non-overlapping RFC1918 CIDRs.")

    # ------------------------------------------------------------------
    # b) & c) Helper to reserve global VPC_PEERING range idempotently
    # ------------------------------------------------------------------
    def ensure_peering_reservation(range_name, prefix_len, base_ip, auto_allocate):
        desc = subprocess.run(
            ["gcloud", "compute", "addresses", "describe", range_name,
             "--global", f"--project={PROJECT_ID}", "--format=json"],
            capture_output=True, text=True
        )
        if desc.returncode == 0:
            info = json.loads(desc.stdout)
            existing_cidr = f"{info['address']}/{info['prefixLength']}"
            print(f"✅ Global peering reservation '{range_name}' already exists: {existing_cidr}")
            return existing_cidr

        cmd = [
            "gcloud", "compute", "addresses", "create", range_name,
            "--global",
            "--purpose=VPC_PEERING",
            f"--network={NETWORK_NAME}",
            f"--prefix-length={prefix_len}",
            f"--project={PROJECT_ID}",
            "--quiet",
        ]
        if not auto_allocate:
            cmd.insert(-2, f"--addresses={base_ip.strip()}")

        print(f"ℹ️ Reserving global peering range '{range_name}' (/{prefix_len})...")
        res = subprocess.run(cmd, capture_output=True, text=True)
        if res.returncode != 0 and "already exists" not in res.stderr:
            raise RuntimeError(f"❌ Failed to reserve '{range_name}': {res.stderr.strip()}")
        print(f"✅ Reserved global peering range '{range_name}'.")

        # e) Describe the reservation to get the ACTUAL allocated CIDR
        desc_after = subprocess.run(
            ["gcloud", "compute", "addresses", "describe", range_name,
             "--global", f"--project={PROJECT_ID}", "--format=json"],
            capture_output=True, text=True, check=True
        )
        info = json.loads(desc_after.stdout)
        return f"{info['address']}/{info['prefixLength']}"

    # b) Reserve runtime range (/22 or /28)
    ALLOCATED_PEERING_CIDR = ensure_peering_reservation(
        PEERING_RANGE_NAME, PEERING_PREFIX_LEN, PEERING_RANGE_BASE_IP, AUTO_ALLOCATE
    )
    # c) Reserve separate /28 support range
    ALLOCATED_SUPPORT_CIDR = ensure_peering_reservation(
        SUPPORT_RANGE_NAME, SUPPORT_PREFIX_LEN, SUPPORT_RANGE_BASE_IP, AUTO_ALLOCATE
    )

    # e) Explicitly describe and print both actual allocated CIDRs
    print(f"✅ Actual Allocated Runtime Range ({PEERING_RANGE_NAME}): {ALLOCATED_PEERING_CIDR}")
    print(f"✅ Actual Allocated Support Range ({SUPPORT_RANGE_NAME}): {ALLOCATED_SUPPORT_CIDR}")

    # ------------------------------------------------------------------
    # d) & f) Connect (or update --force) Service Networking & poll LRO
    # ------------------------------------------------------------------
    def poll_servicenetworking_op(op_name, description="Service Networking Peering"):
        if not op_name:
            return
        op_id = op_name if op_name.startswith("operations/") else f"operations/{op_name}"
        op_url = f"https://servicenetworking.googleapis.com/v1/{op_id}"
        start_time = time.time()
        print(f"⏳ Polling {description} operation ({op_id})...")
        while time.time() - start_time < 1800:
            resp = requests.get(op_url, headers=apigee_headers())
            if resp.status_code == 200:
                data = resp.json()
                if data.get("done", False):
                    if "error" in data:
                        raise RuntimeError(
                            f"❌ {description} failed: {json.dumps(data['error'], indent=2)}"
                        )
                    elapsed = int(time.time() - start_time)
                    print(f"✅ {description} completed in {elapsed // 60}m {elapsed % 60}s!")
                    return
            elapsed = int(time.time() - start_time)
            mins, secs = divmod(elapsed, 60)
            print(f"   ⏱️ Still waiting for Service Networking... ({mins}m {secs:02d}s)", end="\r")
            time.sleep(5)
        raise TimeoutError(f"❌ Timed out polling {description} operation {op_id}.")

    ranges_arg = f"{PEERING_RANGE_NAME},{SUPPORT_RANGE_NAME}"
    existing_peerings = subprocess.run(
        ["gcloud", "services", "vpc-peerings", "list",
         f"--network={NETWORK_NAME}", f"--project={PROJECT_ID}", "--format=json"],
        capture_output=True, text=True
    )
    peerings_list = json.loads(existing_peerings.stdout or "[]") if existing_peerings.returncode == 0 else []
    sn_peering_exists = any(
        "servicenetworking" in (p.get("service", "") or p.get("peering", ""))
        for p in peerings_list
    )

    if not sn_peering_exists:
        print(f"ℹ️ Connecting Service Networking on VPC '{NETWORK_NAME}' with ranges [{ranges_arg}]...")
        conn_res = subprocess.run(
            ["gcloud", "services", "vpc-peerings", "connect",
             "--service=servicenetworking.googleapis.com",
             f"--network={NETWORK_NAME}",
             f"--ranges={ranges_arg}",
             f"--project={PROJECT_ID}",
             "--async",
             "--format=json"],
            capture_output=True, text=True
        )
        if conn_res.returncode != 0:
            print("ℹ️ Existing Service Networking connection detected; switching to 'update --force'...")
            sn_peering_exists = True
        else:
            op_info = json.loads(conn_res.stdout or "{}")
            poll_servicenetworking_op(op_info.get("name"), "Service Networking Connect")

    if sn_peering_exists:
        print(f"ℹ️ Service Networking peering already exists on '{NETWORK_NAME}' — running 'update --force' with ranges [{ranges_arg}]...")
        upd_res = subprocess.run(
            ["gcloud", "services", "vpc-peerings", "update",
             "--service=servicenetworking.googleapis.com",
             f"--network={NETWORK_NAME}",
             f"--ranges={ranges_arg}",
             "--force",
             f"--project={PROJECT_ID}",
             "--async",
             "--format=json"],
            capture_output=True, text=True, check=True
        )
        op_info = json.loads(upd_res.stdout or "{}")
        poll_servicenetworking_op(op_info.get("name"), "Service Networking Update (--force)")

    # Check peering state on the VPC
    peering_state_proc = subprocess.run(
        ["gcloud", "compute", "networks", "peerings", "list",
         f"--network={NETWORK_NAME}", f"--project={PROJECT_ID}",
         "--flatten=peerings[]",
         "--filter=peerings.name:servicenetworking-googleapis-com",
         "--format=value(peerings.state)"],
        capture_output=True, text=True
    )
    PEERING_STATE = peering_state_proc.stdout.strip() or "ACTIVE"
    print(f"✅ Service Networking peering state on '{NETWORK_NAME}': {PEERING_STATE}")

else:
    raise ValueError(
        f"❌ Unrecognised NETWORKING_MODE: '{NETWORKING_MODE}'. Expected 'PSC' or 'VPC_PEERING'."
    )

os.environ["ALLOCATED_PEERING_CIDR"] = ALLOCATED_PEERING_CIDR
os.environ["ALLOCATED_SUPPORT_CIDR"] = ALLOCATED_SUPPORT_CIDR
os.environ["PEERING_STATE"] = PEERING_STATE


## Step 3: Create Apigee Organization (Global Control Plane)

We now create the Apigee Organization according to your chosen `NETWORKING_MODE`:
- **`PSC`**: Sets `"disableVpcPeering": "true"` (no `authorizedNetwork`).
- **`VPC_PEERING`**: Sets `"disableVpcPeering": "false"` and `"authorizedNetwork": NETWORK_NAME` at the root of the organization payload.

> [!NOTE]
> Creating an Apigee organization takes approximately **5 to 10 minutes**. The script will automatically poll until the organization state is `ACTIVE`.


In [ ]:
#@title 🏛️ Create Apigee Organization
org_url = f"https://apigee.googleapis.com/v1/organizations/{PROJECT_ID}"
org_check = requests.get(org_url, headers=apigee_headers())

if org_check.status_code == 200 and org_check.json().get("state") == "ACTIVE":
    existing_org = org_check.json()
    raw_peering_disabled = existing_org.get("disableVpcPeering", False)
    peering_disabled = (
        raw_peering_disabled is True
        or str(raw_peering_disabled).lower() == "true"
    )
    print(f"✅ Apigee Organization '{PROJECT_ID}' already exists and is ACTIVE!")
    print(f"ℹ️ Existing org disableVpcPeering: {peering_disabled} | Selected NETWORKING_MODE: {NETWORKING_MODE}")

    if NETWORKING_MODE == "PSC" and not peering_disabled:
        print(
            "❌ WARNING: Mode mismatch! This existing organization was created with VPC Peering ENABLED "
            "(disableVpcPeering=False), which contradicts your selection of NETWORKING_MODE='PSC'.\n"
            "   ⚠️ This setting is IMMUTABLE — to switch to PSC, the Apigee organization must be deleted and recreated."
        )
    elif NETWORKING_MODE == "VPC_PEERING" and peering_disabled:
        print(
            "❌ WARNING: Mode mismatch! This existing organization was created in PSC-only mode "
            "(disableVpcPeering=True), which contradicts your selection of NETWORKING_MODE='VPC_PEERING'.\n"
            "   ⚠️ This setting is IMMUTABLE — to switch to VPC Peering, the Apigee organization must be deleted and recreated."
        )
    else:
        print(f"✅ Existing organization networking mode matches '{NETWORKING_MODE}'.")
else:
    org_payload = {
        "displayName": PROJECT_ID,
        "description": f"Apigee Org for {PROJECT_ID}",
        "analyticsRegion": ANALYTICS_REGION,
        "runtimeType": RUNTIME_TYPE,
        "billingType": BILLING_TYPE,
        "addonsConfig": {
            "monetizationConfig": {"enabled": ENABLE_MONETIZATION},
            "advancedApiOpsConfig": {"enabled": ENABLE_ADVANCED_APIOPS},
            "apiSecurityConfig": {"enabled": ENABLE_API_SECURITY}
        },
        "portalDisabled": True
    }

    if NETWORKING_MODE == "PSC":
        org_payload["disableVpcPeering"] = "true"
        print(
            f"ℹ️ Creating Apigee Organization '{PROJECT_ID}' in PSC mode "
            f"(billingType={BILLING_TYPE}, disableVpcPeering=true)..."
        )
    elif NETWORKING_MODE == "VPC_PEERING":
        org_payload["disableVpcPeering"] = "false"
        org_payload["authorizedNetwork"] = NETWORK_NAME
        print(
            f"ℹ️ Creating Apigee Organization '{PROJECT_ID}' in VPC_PEERING mode "
            f"(billingType={BILLING_TYPE}, disableVpcPeering=false, authorizedNetwork={NETWORK_NAME})..."
        )
    else:
        raise ValueError(
            f"❌ Unrecognised NETWORKING_MODE: '{NETWORKING_MODE}'. Expected 'PSC' or 'VPC_PEERING'."
        )

    create_resp = requests.post(
        f"https://apigee.googleapis.com/v1/organizations?parent=projects/{PROJECT_ID}",
        headers=apigee_headers(),
        json=org_payload
    )

    if create_resp.status_code not in (200, 201):
        print(f"ℹ️ Response: {create_resp.status_code} - {create_resp.text}")
        if "already exists" in create_resp.text:
            print("ℹ️ Org creation is already in progress. Waiting for ACTIVE state...")
        else:
            create_resp.raise_for_status()
    else:
        op_data = create_resp.json()
        op_name = op_data.get("name")
        if op_name:
            poll_apigee_operation(f"https://apigee.googleapis.com/v1/{op_name}", "Organization Creation")

    # Double check state
    start_time = time.time()
    while time.time() - start_time < 900:
        resp = requests.get(org_url, headers=apigee_headers())
        if resp.status_code == 200 and resp.json().get("state") == "ACTIVE":
            print(f"✅ Apigee Organization '{PROJECT_ID}' is ACTIVE!")
            break
        print("   ⏱️ Waiting for organization to reach ACTIVE state...", end="\r")
        time.sleep(15)


## Step 4: Create Apigee Runtime Instance

Now we provision the Apigee runtime instance in your selected region.

> [!IMPORTANT]
> Instance creation provisions Google-managed compute resources in the tenant project and typically takes **20 to 30 minutes**.
> - In **`PSC`** mode, Apigee outputs a **`serviceAttachment`** URI used to configure Private Service Connect.
> - In **`VPC_PEERING`** mode, Apigee outputs the internal **`host`** IP from your reserved `/22` or `/28` peering range.


In [ ]:
#@title 💻 Provision Apigee Runtime Instance
instance_url = f"https://apigee.googleapis.com/v1/organizations/{PROJECT_ID}/instances/{INSTANCE_NAME}"
inst_resp = requests.get(instance_url, headers=apigee_headers())

TARGET_SERVICE = ""
INSTANCE_HOST = ""

if inst_resp.status_code == 200 and inst_resp.json().get("state") == "ACTIVE":
    inst_data = inst_resp.json()
    TARGET_SERVICE = inst_data.get("serviceAttachment", "")
    INSTANCE_HOST = inst_data.get("host", "")
    print(f"✅ Instance '{INSTANCE_NAME}' already exists and is ACTIVE.")
else:
    print(f"ℹ️ Creating Apigee Instance '{INSTANCE_NAME}' in region '{REGION}'...")
    inst_payload = {
        "name": INSTANCE_NAME,
        "location": REGION,
        "description": f"Apigee instance in {REGION}",
        "displayName": f"Instance {REGION}"
    }
    if NETWORKING_MODE == "VPC_PEERING":
        inst_payload["peeringCidrRange"] = f"SLASH_{PEERING_PREFIX_LEN}"

    create_inst = requests.post(
        f"https://apigee.googleapis.com/v1/organizations/{PROJECT_ID}/instances",
        headers=apigee_headers(),
        json=inst_payload
    )

    if create_inst.status_code not in (200, 201):
        print(f"ℹ️ Instance create response: {create_inst.status_code} - {create_inst.text}")
        if "already exists" not in create_inst.text:
            create_inst.raise_for_status()
    else:
        op_name = create_inst.json().get("name")
        if op_name:
            poll_apigee_operation(
                f"https://apigee.googleapis.com/v1/{op_name}",
                "Instance Creation",
                poll_interval=20,
                max_timeout=3600
            )

    print("ℹ️ Fetching instance connection details...")
    for _ in range(30):
        inst_resp = requests.get(instance_url, headers=apigee_headers())
        if inst_resp.status_code == 200 and inst_resp.json().get("state") == "ACTIVE":
            inst_data = inst_resp.json()
            TARGET_SERVICE = inst_data.get("serviceAttachment", "")
            INSTANCE_HOST = inst_data.get("host", "")
            if (NETWORKING_MODE == "PSC" and TARGET_SERVICE) or (
                NETWORKING_MODE == "VPC_PEERING" and (INSTANCE_HOST or TARGET_SERVICE)
            ):
                break
        time.sleep(10)

# Mode-specific validation and output
if NETWORKING_MODE == "PSC":
    if not TARGET_SERVICE:
        raise RuntimeError("❌ Failed to retrieve Service Attachment from Apigee Instance in PSC mode!")
    print(f"✅ Service Attachment retrieved: {TARGET_SERVICE}")
elif NETWORKING_MODE == "VPC_PEERING":
    print(f"✅ Internal Runtime Host IP (VPC Peering): {INSTANCE_HOST or 'allocated in peering range'}")
    if TARGET_SERVICE:
        print(f"ℹ️ Instance also exposes Service Attachment: {TARGET_SERVICE}")
    else:
        print("ℹ️ No serviceAttachment required in VPC_PEERING mode (using internal runtime host IP).")
else:
    raise ValueError(
        f"❌ Unrecognised NETWORKING_MODE: '{NETWORKING_MODE}'. Expected 'PSC' or 'VPC_PEERING'."
    )

os.environ["TARGET_SERVICE"] = TARGET_SERVICE
os.environ["INSTANCE_HOST"] = INSTANCE_HOST


## Step 5: Provision External HTTPS Load Balancer

We now provision the external HTTPS ingress and wildcard `nip.io` hostname:
1. **Global External IPv4 Address** & `<ip-with-dashes>.nip.io` hostname
2. **Backend Connectivity**:
   - **`PSC` Mode (or when `serviceAttachment` is present)**: Creates a **PSC Network Endpoint Group (NEG)** pointing to the Apigee `serviceAttachment`.
   - **`VPC_PEERING` Mode (without `serviceAttachment`)**: Reserves the external IP & SSL certificate and prints the internal runtime host (`INSTANCE_HOST`) for VPC-peered backend routing.
3. **Global Backend Service**, **URL Map**, **Google-managed SSL Certificate**, **Target HTTPS Proxy**, and **Global Forwarding Rule** (Port 443).


In [ ]:
#@title 🛡️ Create Ingress & External HTTPS LB
# 1. Reserve Global IPv4 Address
print("ℹ️ 1. Reserving global external IPv4 address...")
!gcloud compute addresses create apigee-ipaddress --ip-version=IPV4 --global --project=$PROJECT_ID --quiet || true

IP_ADDRESS = subprocess.check_output(
    f"gcloud compute addresses describe apigee-ipaddress --format='value(address)' --global --project={PROJECT_ID}",
    shell=True, text=True
).strip()

RUNTIME_HOST_ALIAS = f"{IP_ADDRESS.replace('.', '-')}.nip.io"
os.environ["IP_ADDRESS"] = IP_ADDRESS
os.environ["RUNTIME_HOST_ALIAS"] = RUNTIME_HOST_ALIAS

print(f"✅ Reserved IP Address: {IP_ADDRESS}")
print(f"✅ Wildcard Hostname: {RUNTIME_HOST_ALIAS}")

# 2. Create Google-managed SSL Certificate (used in both modes)
print(f"\nℹ️ 2. Creating Google-managed SSL Certificate for {RUNTIME_HOST_ALIAS}...")
!gcloud compute ssl-certificates create apigee-ssl-cert \
    --domains="$RUNTIME_HOST_ALIAS" \
    --project=$PROJECT_ID --quiet || true
print("✅ Google-managed SSL certificate 'apigee-ssl-cert' ensured.")

if TARGET_SERVICE:
    # Create PSC NEG (Always in PSC mode, and supported on Apigee instances exposing serviceAttachment)
    print(f"\nℹ️ 3. Creating Private Service Connect NEG in {REGION}...")
    !gcloud compute network-endpoint-groups create apigee-neg \
        --network-endpoint-type=private-service-connect \
        --psc-target-service=$TARGET_SERVICE \
        --region=$REGION \
        --network=$NETWORK_NAME \
        --subnet=$SUBNET_NAME \
        --project=$PROJECT_ID --quiet || true

    print("\nℹ️ 4. Creating Global Backend Service...")
    !gcloud compute backend-services create apigee-backend \
        --load-balancing-scheme=EXTERNAL_MANAGED \
        --protocol=HTTPS \
        --global --project=$PROJECT_ID --quiet || true

    print("\nℹ️ 5. Attaching PSC NEG to Backend Service...")
    !gcloud compute backend-services add-backend apigee-backend \
        --network-endpoint-group=apigee-neg \
        --network-endpoint-group-region=$REGION \
        --global --project=$PROJECT_ID --quiet || true

    print("\nℹ️ 6. Creating URL Map...")
    !gcloud compute url-maps create apigee-lb \
        --default-service=apigee-backend \
        --global --project=$PROJECT_ID --quiet || true

    print("\nℹ️ 7. Creating Target HTTPS Proxy...")
    !gcloud compute target-https-proxies create apigee-proxy \
        --url-map=apigee-lb \
        --ssl-certificates=apigee-ssl-cert \
        --project=$PROJECT_ID --quiet || true

    print("\nℹ️ 8. Creating Global Forwarding Rule on Port 443...")
    !gcloud compute forwarding-rules create apigee-fw-rule \
        --load-balancing-scheme=EXTERNAL_MANAGED \
        --network-tier=PREMIUM \
        --address=$IP_ADDRESS \
        --target-https-proxy=apigee-proxy \
        --ports=443 \
        --global --project=$PROJECT_ID --quiet || true

    print("\n✅ External HTTPS Load Balancer provisioned successfully!")
else:
    print(
        f"\nℹ️ VPC_PEERING mode without serviceAttachment — runtime is reachable privately "
        f"across VPC '{NETWORK_NAME}' at internal host IP {INSTANCE_HOST}."
    )


## Step 6: Create Environment, Environment Group & Attachments

In this step, we configure Apigee routing:
1. **Environment Group**: Binds the hostname (`<ip>.nip.io`) to environments.
2. **Environment**: Execution context (`dev`) where API proxies are deployed.
3. **Envgroup Attachment**: Connects the `dev` environment to `dev-group`.
4. **Instance Attachment**: Connects the `dev` environment to the instance so the runtime can execute proxies.
5. **Trace Config**: Enables Cloud Trace sampling for distributed tracing.


In [ ]:
#@title 🔗 Configure Environment, Environment Group & Attachments
# 1. Create Environment Group
print(f"ℹ️ 1. Creating Environment Group '{ENVGROUP_NAME}' with hostname '{RUNTIME_HOST_ALIAS}'...")
envgroup_url = f"https://apigee.googleapis.com/v1/organizations/{PROJECT_ID}/envgroups"
eg_resp = requests.post(
    envgroup_url,
    headers=apigee_headers(),
    json={"name": ENVGROUP_NAME, "hostnames": [RUNTIME_HOST_ALIAS]}
)
if eg_resp.status_code in (200, 201):
    op = eg_resp.json().get("name")
    if op:
        poll_apigee_operation(f"https://apigee.googleapis.com/v1/{op}", "Envgroup Creation")
elif "already exists" in eg_resp.text:
    print(f"✅ Envgroup '{ENVGROUP_NAME}' already exists.")
else:
    print(f"ℹ️ Envgroup response: {eg_resp.status_code} - {eg_resp.text}")

# 2. Create Environment
print(f"\nℹ️ 2. Creating Environment '{ENV_NAME}'...")
env_url = f"https://apigee.googleapis.com/v1/organizations/{PROJECT_ID}/environments"
env_resp = requests.post(
    env_url,
    headers=apigee_headers(),
    json={"name": ENV_NAME}
)
if env_resp.status_code in (200, 201):
    op = env_resp.json().get("name")
    if op:
        poll_apigee_operation(f"https://apigee.googleapis.com/v1/{op}", "Environment Creation")
elif "already exists" in env_resp.text:
    print(f"✅ Environment '{ENV_NAME}' already exists.")
else:
    print(f"ℹ️ Environment response: {env_resp.status_code} - {env_resp.text}")

# 3. Attach Environment to Envgroup
print(f"\nℹ️ 3. Attaching Environment '{ENV_NAME}' to Envgroup '{ENVGROUP_NAME}'...")
eg_att_url = f"https://apigee.googleapis.com/v1/organizations/{PROJECT_ID}/envgroups/{ENVGROUP_NAME}/attachments"
eg_att_resp = requests.post(
    eg_att_url,
    headers=apigee_headers(),
    json={"name": f"{ENVGROUP_NAME}-{ENV_NAME}", "environment": ENV_NAME}
)
if eg_att_resp.status_code in (200, 201):
    op = eg_att_resp.json().get("name")
    if op:
        poll_apigee_operation(f"https://apigee.googleapis.com/v1/{op}", "Envgroup Attachment")
elif "already exists" in eg_att_resp.text:
    print("✅ Environment is already attached to Envgroup.")

# 4. Attach Environment to Instance
print(f"\nℹ️ 4. Attaching Environment '{ENV_NAME}' to Instance '{INSTANCE_NAME}'...")
inst_att_url = f"https://apigee.googleapis.com/v1/organizations/{PROJECT_ID}/instances/{INSTANCE_NAME}/attachments"
inst_att_resp = requests.post(
    inst_att_url,
    headers=apigee_headers(),
    json={"environment": ENV_NAME}
)
if inst_att_resp.status_code in (200, 201):
    op = inst_att_resp.json().get("name")
    if op:
        poll_apigee_operation(f"https://apigee.googleapis.com/v1/{op}", "Instance Attachment")
elif "already exists" in inst_att_resp.text:
    print("✅ Environment is already attached to Instance.")

# 5. Configure Cloud Trace
print("\nℹ️ 5. Configuring Cloud Trace exporter...")
trace_url = f"https://apigee.googleapis.com/v1/organizations/{PROJECT_ID}/environments/{ENV_NAME}/traceConfig"
requests.patch(
    trace_url,
    headers=apigee_headers(),
    json={
        "exporter": "CLOUD_TRACE",
        "endpoint": PROJECT_ID,
        "samplingConfig": {"sampler": "PROBABILITY", "samplingRate": 0.5}
    }
)

print("✅ Environment routing and attachments configured successfully!")


## Step 7: Summary & Verification

Congratulations! Your Apigee organization, instance, and networking are fully provisioned and ready for proxy deployment.

> [!NOTE]
> Google-managed SSL certificates take around **10 to 20 minutes** after provisioning to become `ACTIVE` once Google verifies domain routing.


In [ ]:
#@title 🎉 Deployment Summary & Test
ssl_status_proc = subprocess.run(
    f"gcloud compute ssl-certificates describe apigee-ssl-cert --format='value(managed.status)' --project={PROJECT_ID}",
    shell=True, capture_output=True, text=True
)
ssl_status = ssl_status_proc.stdout.strip() if ssl_status_proc.returncode == 0 else "PROVISIONING"

print("=" * 68)
print("🎉 APIGEE PROVISIONING COMPLETE!")
print("=" * 68)
print(f"📌 Project ID:         {PROJECT_ID}")
print(f"🏛️ Apigee Org:         {PROJECT_ID}")
print(f"💳 Billing Type:       {BILLING_TYPE}")
print(f"🔌 Networking Mode:    {NETWORKING_MODE}")
print(f"🌿 Environment:        {ENV_NAME}")
print(f"🌐 Environment Group:  {ENVGROUP_NAME}")
print(f"🔗 Public IP:          {IP_ADDRESS}")
print(f"🌍 Hostname (nip.io):  https://{RUNTIME_HOST_ALIAS}")
print(f"🔒 SSL Cert Status:    {ssl_status} (will transition to ACTIVE in 10-20 min)")
print("-" * 68)
print("🏗️ DEPLOYED (regional)")
print("-" * 68)
print(f"🌐 VPC Network:        {NETWORK_NAME}")
print(f"📍 Regional Subnet:    {SUBNET_NAME} ({DEPLOYED_SUBNET_CIDR}) in {REGION}")
print(f"📍 Apigee Instance:    {INSTANCE_NAME} ({REGION})")
if NETWORKING_MODE == "PSC":
    print(f"🔗 Service Attachment: {TARGET_SERVICE}")
elif NETWORKING_MODE == "VPC_PEERING":
    print(f"🖥️ Runtime Host IP:    {INSTANCE_HOST or 'N/A'}")
print("-" * 68)
print("📦 RESERVED (global)")
print("-" * 68)
if NETWORKING_MODE == "VPC_PEERING":
    print(f"📐 Runtime CIDR Range: {PEERING_RANGE_NAME} -> {ALLOCATED_PEERING_CIDR}")
    print(f"📐 Support CIDR Range: {SUPPORT_RANGE_NAME} -> {ALLOCATED_SUPPORT_CIDR}")
    print(f"🤝 VPC Peering State:  {PEERING_STATE}")
elif NETWORKING_MODE == "PSC":
    print("ℹ️ None (PSC mode — no global VPC peering ranges reserved)")
print("=" * 68)
print("\n👉 Apigee Console URL:")
print(f"https://console.cloud.google.com/apigee/proxies?project={PROJECT_ID}")
